In [ ]:
# ============================================================
# GLUCOTWIN AI - COMPLETE DATASET CREATION
# Creates:
# 1. glucotwin_combined_dataset.csv
# 2. glucotwin_ehr_data.csv
# 3. glucotwin_wearable_data.csv
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files

# Reproducibility
np.random.seed(42)

# ============================================================
# SETTINGS
# ============================================================

NUM_PATIENTS = 500
DAYS = 7
HOURS_PER_PATIENT = DAYS * 24

rows = []

# ============================================================
# GENERATE SYNTHETIC PATIENT DATA
# ============================================================

for patient_id in range(1, NUM_PATIENTS + 1):

    # -------------------------
    # Static EHR information
    # -------------------------

    age = np.random.randint(30, 71)

    gender = np.random.choice(
        ["Male", "Female"]
    )

    bmi = round(
        np.clip(
            np.random.normal(27.5, 4.2),
            18,
            39.7
        ),
        1
    )

    hba1c = round(
        np.clip(
            np.random.normal(6.24, 0.82),
            4.5,
            8.7
        ),
        1
    )

    systolic_bp = int(
        np.clip(
            np.random.normal(135, 14.5),
            110,
            160
        )
    )

    diabetes_history = np.random.choice(
        [0, 1],
        p=[0.65, 0.35]
    )

    family_history = np.random.choice(
        [0, 1],
        p=[0.55, 0.45]
    )

    # -------------------------
    # Patient baseline glucose
    # -------------------------

    base_glucose = (
        115
        + (diabetes_history * 12)
        + ((hba1c - 5.5) * 8)
        + ((bmi - 25) * 0.5)
    )

    glucose_values = []

    # ========================================================
    # GENERATE 7 DAYS OF HOURLY GLUCOSE
    # ========================================================

    for hour_index in range(HOURS_PER_PATIENT):

        hour = hour_index % 24

        # Meal-related synthetic effects
        breakfast_effect = (
            12 if 7 <= hour <= 9 else 0
        )

        lunch_effect = (
            15 if 12 <= hour <= 14 else 0
        )

        dinner_effect = (
            18 if 19 <= hour <= 21 else 0
        )

        meal_effect = (
            breakfast_effect
            + lunch_effect
            + dinner_effect
        )

        # Random variation
        noise = np.random.normal(0, 7)

        if hour_index == 0:
            previous_glucose = base_glucose
        else:
            previous_glucose = glucose_values[-1]

        glucose = (
            0.65 * previous_glucose
            + 0.35 * (
                base_glucose
                + meal_effect
            )
            + noise
        )

        glucose = np.clip(
            glucose,
            70,
            230
        )

        glucose_values.append(
            glucose
        )

    # ========================================================
    # CREATE RECORDS
    # ========================================================

    start_time = (
        pd.Timestamp("2026-01-01")
        + pd.Timedelta(
            days=(patient_id - 1) % 30
        )
    )

    for hour_index in range(
        HOURS_PER_PATIENT - 2
    ):

        timestamp = (
            start_time
            + pd.Timedelta(
                hours=hour_index
            )
        )

        current_glucose = (
            glucose_values[hour_index]
        )

        future_glucose = (
            glucose_values[
                hour_index + 2
            ]
        )

        # -------------------------
        # Wearable signals
        # -------------------------

        heart_rate = np.clip(
            np.random.normal(
                70,
                5.4
            ),
            55,
            92
        )

        steps = int(
            np.clip(
                np.random.gamma(
                    2.2,
                    290
                ),
                20,
                1800
            )
        )

        sleep_hours = round(
            np.clip(
                np.random.normal(
                    6.5,
                    1.18
                ),
                4,
                9
            ),
            1
        )

        # ====================================================
        # TARGET
        # ====================================================

        # Synthetic research threshold:
        # A glucose increase of >=25 mg/dL
        # within the next 2 hours is labelled as a spike.

        glucose_spike = int(
            future_glucose
            >= current_glucose + 25
        )

        rows.append([
            patient_id,
            timestamp,
            round(current_glucose, 1),
            round(heart_rate, 1),
            steps,
            sleep_hours,
            round(future_glucose, 1),
            glucose_spike,
            age,
            gender,
            bmi,
            hba1c,
            systolic_bp,
            diabetes_history,
            family_history
        ])

# ============================================================
# COMBINED DATASET
# ============================================================

columns = [
    "patient_id",
    "timestamp",
    "glucose",
    "heart_rate",
    "steps",
    "sleep_hours",
    "glucose_2h_future",
    "glucose_spike",
    "age",
    "gender",
    "bmi",
    "hba1c",
    "systolic_bp",
    "diabetes_history",
    "family_history"
]

df = pd.DataFrame(
    rows,
    columns=columns
)

# Sort properly
df = df.sort_values(
    ["patient_id", "timestamp"]
).reset_index(drop=True)

# ============================================================
# EHR DATASET
# ============================================================

ehr_columns = [
    "patient_id",
    "age",
    "gender",
    "bmi",
    "hba1c",
    "systolic_bp",
    "diabetes_history",
    "family_history"
]

ehr_df = (
    df[ehr_columns]
    .drop_duplicates(
        subset=["patient_id"]
    )
    .reset_index(drop=True)
)

# ============================================================
# WEARABLE DATASET
# ============================================================

wearable_columns = [
    "patient_id",
    "timestamp",
    "glucose",
    "heart_rate",
    "steps",
    "sleep_hours"
]

wearable_df = (
    df[wearable_columns]
    .copy()
)

# ============================================================
# SAVE CSV FILES
# ============================================================

combined_file = (
    "glucotwin_combined_dataset.csv"
)

ehr_file = (
    "glucotwin_ehr_data.csv"
)

wearable_file = (
    "glucotwin_wearable_data.csv"
)

df.to_csv(
    combined_file,
    index=False
)

ehr_df.to_csv(
    ehr_file,
    index=False
)

wearable_df.to_csv(
    wearable_file,
    index=False
)

# ============================================================
# DATA QUALITY CHECK
# ============================================================

print("=" * 60)
print("          GLUCOTWIN AI DATASET CREATED")
print("=" * 60)

print()

print("COMBINED DATASET")
print("----------------")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print(
    "Patients:",
    df["patient_id"].nunique()
)

print()

print("EHR DATASET")
print("-----------")
print(
    "Rows:",
    len(ehr_df)
)
print(
    "Columns:",
    len(ehr_df.columns)
)

print()

print("WEARABLE DATASET")
print("----------------")
print(
    "Rows:",
    len(wearable_df)
)
print(
    "Columns:",
    len(wearable_df.columns)
)

print()

print("TARGET DISTRIBUTION")
print("-------------------")

target_counts = (
    df["glucose_spike"]
    .value_counts()
    .sort_index()
)

print(target_counts)

print()

print("TARGET PERCENTAGE")
print("------------------")

target_percentage = (
    df["glucose_spike"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    * 100
)

print(
    target_percentage.round(2)
)

print()

print("MISSING VALUES")
print("--------------")

print(
    "Total missing:",
    df.isnull().sum().sum()
)

print()

print("DUPLICATES")
print("----------")

print(
    "Duplicate rows:",
    df.duplicated().sum()
)

print()

print("FILES CREATED")
print("-------------")

print("✓", combined_file)
print("✓", ehr_file)
print("✓", wearable_file)

print()

print("=" * 60)
print("DOWNLOAD STARTING...")
print("=" * 60)

# ============================================================
# DOWNLOAD FILES
# ============================================================

files.download(
    combined_file
)

files.download(
    ehr_file
)

files.download(
    wearable_file
)

print()
print("✓ All 3 CSV files are ready.")